# Colab 2 · Orchestration Patterns in Depth
### Day 19 — Agent Orchestration with AutoGen Studio & Semantic Kernel

Colab 1 used the simplest pattern (RoundRobin). Now you'll wire the **same research team four different ways** and watch how the *control flow* changes — then peek at the **same idea in Semantic Kernel**.

**You will build:**
1. **SelectorGroupChat** — an LLM decides who speaks next.
2. **Swarm** — agents hand off to each other directly.
3. **GraphFlow** — a deterministic researcher → writer → reviewer graph.
4. A **function tool** the researcher calls to delegate real work.
5. A **Semantic Kernel** sequential-orchestration mini-example.

⏱️ ~60 min including the extension tasks at the end.

> The patterns are the lesson. AutoGen, Semantic Kernel and the Microsoft Agent Framework all expose this same family — RoundRobin/Sequential, Selector/GroupChat, Swarm/Handoff, Graph, Magentic.

## 0 · Setup

In [1]:
%pip install -q -U "autogen-agentchat" "autogen-ext[openai]"
print("AutoGen installed.")

Note: you may need to restart the kernel to use updated packages.
AutoGen installed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogenstudio 0.4.2.2 requires autogen-agentchat<0.6,>=0.4.9.2, but you have autogen-agentchat 0.7.5 which is incompatible.
autogenstudio 0.4.2.2 requires autogen-core<0.6,>=0.4.9.2, but you have autogen-core 0.7.5 which is incompatible.
autogenstudio 0.4.2.2 requires autogen-ext[azure,magentic-one,openai]<0.6,>=0.4.2, but you have autogen-ext 0.7.5 which is incompatible.

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os

# Hardcode the API key
os.environ["OPENAI_API_KEY"] = "<YOUR_OPENAI_API_KEY>"
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import (
    TextMentionTermination,
    MaxMessageTermination,
)
from autogen_agentchat.ui import Console

# Create the model client
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
)

print("Ready.")

Ready.


### Three specialists we'll reuse

Notice the **descriptions** — Selector and Swarm route work based on them, so they have to be specific and non-overlapping.

In [4]:
def make_specialists():
    planner = AssistantAgent(
        name="planner",
        model_client=model_client,
        description="Breaks a topic into 2-3 concrete sub-questions to research.",
        system_message="You plan research. Given a topic, list 2-3 specific sub-questions. Keep it short.",
    )
    researcher = AssistantAgent(
        name="researcher",
        model_client=model_client,
        description="Answers factual sub-questions with concise bullet points.",
        system_message="You answer the planner's sub-questions with short factual bullets.",
    )
    writer = AssistantAgent(
        name="writer",
        model_client=model_client,
        description="Turns research bullets into a tight 4-sentence summary, ending with APPROVE.",
        system_message="Write a tight 4-sentence summary from the research. End your message with APPROVE.",
    )
    return planner, researcher, writer

print("Specialist factory ready.")

Specialist factory ready.


## 1 · SelectorGroupChat — let an LLM route

Instead of a fixed order, a **SelectorGroupChat** uses a model to pick *who should act next* based on the conversation and each agent's `description`. Good when the next best speaker depends on what just happened.

Key knobs: it needs its own `model_client` to do the routing, and `allow_repeated_speaker=False` stops one agent from monopolising the floor.

In [5]:
from autogen_agentchat.teams import SelectorGroupChat

planner, researcher, writer = make_specialists()
termination = TextMentionTermination("APPROVE") | MaxMessageTermination(8)

selector_team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,        # the "router" brain
    termination_condition=termination,
    allow_repeated_speaker=False,
)

await Console(selector_team.run_stream(
    task="Topic: why are reusable cups better than disposable ones?"
))

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************U50A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

Look at the speaker order in the transcript — it was **chosen at runtime**, not fixed. That's the difference from RoundRobin.

## 2 · Swarm — agents hand off to each other

In a **Swarm**, control is decentralised: each agent declares who it can **hand off** to via `handoffs=[...]`, and passes control with a handoff message. There's no central router — the agents themselves decide.

We'll build a tiny triage flow: a `triage` agent routes to either `billing` or `tech`, and those can hand back to the user when done.

In [6]:
from autogen_agentchat.teams import Swarm
from autogen_agentchat.conditions import HandoffTermination

triage = AssistantAgent(
    name="triage",
    model_client=model_client,
    handoffs=["billing", "tech"],
    description="Front desk: routes the user to the right specialist.",
    system_message="Decide if the request is about billing or tech, then hand off to that agent.",
)
billing = AssistantAgent(
    name="billing",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles billing and refund questions.",
    system_message="Answer the billing question. If it's not billing, hand back to triage.",
)
tech = AssistantAgent(
    name="tech",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles technical troubleshooting.",
    system_message="Answer the tech question concisely, then say DONE.",
)

swarm = Swarm(
    participants=[triage, billing, tech],          # Swarm starts with the first agent
    termination_condition=TextMentionTermination("DONE") | MaxMessageTermination(8),
)

await Console(swarm.run_stream(task="My app keeps crashing when I open the camera."))

---------- TextMessage (user) ----------
My app keeps crashing when I open the camera.


Error processing publish message for triage_ac42e1b3-f1bb-4f39-8967-aaac0264ea44/ac42e1b3-f1bb-4f39-8967-aaac0264ea44
Traceback (most recent call last):
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib

RuntimeError: AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************U50A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Traceback:
Traceback (most recent call last):

  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_agentchat\teams\_group_chat\_chat_agent_container.py", line 133, in handle_request
    async for msg in self._agent.on_messages_stream(self._message_buffer, ctx.cancellation_token):

  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py", line 953, in on_messages_stream
    async for inference_output in self._call_llm(

  File "c:\Users

Watch for the **HandoffMessage** in the transcript — that's one agent explicitly delegating to another. `HandoffTermination(target="user")` is another common stop condition when an agent hands control back to a human.

## 3 · GraphFlow — a deterministic workflow

When you need the **same path every time** (auditable, reproducible), use **GraphFlow**. You declare nodes and directed edges with `DiGraphBuilder`; execution follows the graph exactly.

Here: `planner → researcher → writer`, a fixed pipeline with no LLM routing brain.

In [7]:
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow

planner, researcher, writer = make_specialists()

builder = DiGraphBuilder()
builder.add_node(planner).add_node(researcher).add_node(writer)
builder.add_edge(planner, researcher).add_edge(researcher, writer)
graph = builder.build()

flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
)

await Console(flow.run_stream(task="Topic: the benefits of cycling to work."))

---------- TextMessage (user) ----------
Topic: the benefits of cycling to work.


Error processing publish message for planner_7e37795b-267a-4943-8d10-eab8b1574256/7e37795b-267a-4943-8d10-eab8b1574256
Traceback (most recent call last):
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Li

RuntimeError: AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************U50A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Traceback:
Traceback (most recent call last):

  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_agentchat\teams\_group_chat\_chat_agent_container.py", line 133, in handle_request
    async for msg in self._agent.on_messages_stream(self._message_buffer, ctx.cancellation_token):

  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py", line 953, in on_messages_stream
    async for inference_output in self._call_llm(

  File "c:\Users

GraphFlow gives you **determinism**: the order is guaranteed by the graph, not decided by a model. That's exactly what you want for a compliance-sensitive or repeatable pipeline.

## 4 · A function tool the researcher can call

Delegation isn't only agent-to-agent — an agent can delegate to **code** via a tool. Define a plain Python function, pass it in `tools=[...]`, and the agent will call it when useful. (Here it's a stub; swap in a real search API at home.)

In [8]:
def web_search(query: str) -> str:
    """Look up a query and return a short text snippet. (Stub for the workshop.)"""
    canned = {
        "reusable cup co2": "A reusable cup typically breaks even vs. disposables after ~20-100 uses.",
        "default": "No exact match; returning a generic note that reusable goods amortise their footprint with use.",
    }
    return canned.get(query.lower().strip(), canned["default"])

researcher_with_tool = AssistantAgent(
    name="researcher",
    model_client=model_client,
    tools=[web_search],
    description="Researches facts, calling web_search when it needs evidence.",
    system_message="Use the web_search tool to find a figure, then report it in one bullet. End with APPROVE.",
)

from autogen_agentchat.teams import RoundRobinGroupChat
tool_team = RoundRobinGroupChat(
    [researcher_with_tool],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(4),
)
await Console(tool_team.run_stream(task="Find a figure on reusable cup CO2 break-even and report it."))

---------- TextMessage (user) ----------
Find a figure on reusable cup CO2 break-even and report it.


Error processing publish message for researcher_89fcf974-7af1-4c3a-aa0f-b608a3675a74/89fcf974-7af1-4c3a-aa0f-b608a3675a74
Traceback (most recent call last):
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312

RuntimeError: AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************U50A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Traceback:
Traceback (most recent call last):

  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_agentchat\teams\_group_chat\_chat_agent_container.py", line 133, in handle_request
    async for msg in self._agent.on_messages_stream(self._message_buffer, ctx.cancellation_token):

  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py", line 953, in on_messages_stream
    async for inference_output in self._call_llm(

  File "c:\Users

The transcript shows a **ToolCall** and its result — the agent delegated part of its job to your function. In production you'd scope tools to least privilege and validate their arguments.

## 5 · The same idea in Semantic Kernel

Semantic Kernel expresses these patterns too — its **Sequential** orchestration is the SK analogue of RoundRobin/GraphFlow-in-a-line. The code below shows the *shape* of SK agent orchestration.

> ⚠️ SK's agent-orchestration API is newer and evolving (and SK is in maintenance mode heading into the Microsoft Agent Framework). If an import path has moved, check the official Semantic Kernel docs — the **concept** is what transfers, not the exact symbol names.

In [9]:
%pip install -q -U semantic-kernel
print("Semantic Kernel installed.")

Note: you may need to restart the kernel to use updated packages.
Semantic Kernel installed.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# The shape of an SK sequential orchestration: two agents, output of one feeds the next.
# Wrapped in try/except because SK's orchestration symbols move between versions.
import asyncio

try:
    from semantic_kernel.agents import ChatCompletionAgent
    from semantic_kernel.agents.orchestration.sequential import SequentialOrchestration
    from semantic_kernel.agents.runtime import InProcessRuntime
    from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

    service = OpenAIChatCompletion(ai_model_id="gpt-4o-mini")

    sk_writer = ChatCompletionAgent(
        name="writer", service=service,
        instructions="Write one short paragraph on the given topic.",
    )
    sk_editor = ChatCompletionAgent(
        name="editor", service=service,
        instructions="Tighten the paragraph you receive into two crisp sentences.",
    )

    orchestration = SequentialOrchestration(members=[sk_writer, sk_editor])
    runtime = InProcessRuntime()
    runtime.start()

    result = await orchestration.invoke(
        task="The benefits of walking meetings.", runtime=runtime
    )
    print(await result.get())
    await runtime.stop_when_idle()

except Exception as e:
    print("SK orchestration symbols may have moved in your installed version.")
    print("Concept: members=[writer, editor] run in sequence, output -> input.")
    print("Check https://learn.microsoft.com/semantic-kernel for the current API.")
    print("Error was:", type(e).__name__, e)

Exception occurred in agent writer_9710e01b0d0b4f56a6161806a19c2ff9/default:
("<class 'semantic_kernel.connectors.ai.open_ai.services.open_ai_chat_completion.OpenAIChatCompletion'> service failed to complete the prompt", AuthenticationError("Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************U50A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}"))


SK orchestration symbols may have moved in your installed version.
Concept: members=[writer, editor] run in sequence, output -> input.
Check https://learn.microsoft.com/semantic-kernel for the current API.
Error was: ServiceResponseException ("<class 'semantic_kernel.connectors.ai.open_ai.services.open_ai_chat_completion.OpenAIChatCompletion'> service failed to complete the prompt", AuthenticationError("Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************U50A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}"))


Notice the **identical mental model**: a list of agents, run in order, each one's output feeding the next. RoundRobin (AutoGen) ≈ Sequential (SK) ≈ a linear GraphFlow. Learn it once.

---
## 🚀 Extension tasks

### Extension 1 — Write a custom selector function
`SelectorGroupChat` accepts a `selector_func` that overrides the LLM router with your own logic. Write a function that **forces** `planner` to go first, then lets the model choose. (Signature: it receives the message history and returns the next speaker's name, or `None` to defer to the model.)

### Extension 2 — Add a conditional GraphFlow edge
Extend the graph from §3 with a **reviewer** node and a **conditional edge**: if the writer's output contains the word `REVISE`, loop back to the writer; otherwise finish. Use `DiGraphBuilder`'s conditional-edge support and a stop condition so it can't loop forever.

### Extension 3 — Nest a team inside a graph node
A node in a GraphFlow can itself be a **team**. Replace the single `researcher` node with a 2-agent `RoundRobinGroupChat` (researcher + fact-checker) and wire that team in as one node. This is *composition*: patterns nest inside patterns.

Scaffolds below.

In [ ]:
# === Extension 1 scaffold: custom selector_func ===
def force_planner_first(messages):
    # Return an agent name (str) to force a speaker, or None to let the model decide.
    if len(messages) <= 1:
        return "planner"
    return None

planner, researcher, writer = make_specialists()
team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,
    selector_func=force_planner_first,
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(8),
)
await Console(team.run_stream(task="Topic: benefits of a standing desk."))

In [ ]:
# === Extension 2 scaffold: conditional edge (writer loops on REVISE) ===
builder = DiGraphBuilder()
builder.add_node(planner).add_node(researcher).add_node(writer).add_node(reviewer)
builder.add_edge(planner, researcher).add_edge(researcher, writer).add_edge(writer, reviewer)
# Conditional: reviewer -> writer only if "REVISE" appears; else the run ends.
builder.add_edge(reviewer, writer, condition=lambda msg: "REVISE" in msg.to_text())
graph = builder.build()
flow = GraphFlow(participants=builder.get_participants(), graph=graph)

In [ ]:
# === Extension 3 scaffold: nest a team inside a node ===
research_team = RoundRobinGroupChat(
    [researcher, fact_checker],
    termination_condition=MaxMessageTermination(4),
)
builder = DiGraphBuilder()
builder.add_node(planner).add_node(research_team).add_node(writer)
builder.add_edge(planner, research_team).add_edge(research_team, writer)

## Recap

You orchestrated one research team **five ways** and saw exactly how control flow differs:

* **Selector** — LLM picks the next speaker (dynamic).
* **Swarm** — agents hand off to each other (decentralised).
* **GraphFlow** — a fixed, auditable graph (deterministic).
* **Tools** — an agent delegates work to a function.
* **Semantic Kernel** — the same Sequential idea in the enterprise SDK.

Pair this with the decision matrix from the slides, then tackle the **capstone**: build your own delegating team in AutoGen Studio.

In [ ]:
await model_client.close()
print("Client closed. On to the capstone!")